In [ ]:
"""
Simplified spike-data extraction script and firing rate and psth calculating and plotting 

"""

import os
import glob
import json
import pickle
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# paths (edit these if your data locations differ)
# ---------------------------------------------------------------------------
bhv_data_root = "/gpfs/marilyn/pi/nandy/Jamie/marmoset_behavioral_data/"

# neural_data_folder = "/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/"
neural_data_folder = "/gpfs/marilyn/pi/nandy/Jamie/marmoset_neural_recording/"

data_saved_folder = "/gpfs/marilyn/pi/nandy/Jamie/analysis_result_folder/"

fs_spikes = 20000  # neural recording sampling rate (Hz)


In [ ]:

# ---------------------------------------------------------------------------
# session metadata lookup: neural_record_condition -> session info
#
# THIS IS THE ONLY THING YOU SHOULD NEED TO EDIT.
# Keyed by `neural_record_condition` (unique per animal per recording),
# NOT by date -- the same date_tgt can show up twice when two animals are
# recorded in the same session, so it can't be used as a dict key.
# `date_tgt`/`videodate_tgt` are stored as fields instead, for building the
# behavioral file paths.
# `trig_channelname` is the NI trigger channel used to align neural time to
# session start for that recording -- update it freely per session.
# ---------------------------------------------------------------------------

def _build_session_metadata():
    meta = {}

    # ---- Cubone ----
    # the list for neural recording data (from the logger)
    neural_record_conditions = [
        '20260709_Cubone_MC_withDitto',
    ]
    # the list for behavioral data (from the task code)
    dates_list = [
        '20260709_MC',
    ]
    # the list for the video and DLC tracking (from the camera)
    videodates_list = [
        '20260709_MC',
    ]
    # the session start time from the video (when is the beep sound and the first led flash)
    session_start_times = [
        10,
    ]
    trig_channelnames = [
        'Dev1/ai9',
                        ] # transcevier 1  is 'Dev1/ai0', and transceiver is 'Dev1/ai9'

    kilosortvers = [4] * len(dates_list)
    animal1_fixedorders = ['cubone'] * len(dates_list)
    animal2_fixedorders = ['ditto'] * len(dates_list)
    animal1_filenames = ["Cubone"] * len(dates_list)
    animal2_filenames = ["Ditto"] * len(dates_list)
    recorded_animals = ['cubone'] * len(dates_list) # make sure it's cubone


    for i, d in enumerate(neural_record_conditions):
        meta[d] = dict(
            neural_record_condition=neural_record_conditions[i],
            date_tgt=dates_list[i],
            videodate_tgt=videodates_list[i],
            kilosortver=kilosortvers[i],
            session_start_time=session_start_times[i],
            animal1_fixedorder=animal1_fixedorders[i],
            animal2_fixedorder=animal2_fixedorders[i],
            animal1_filename=animal1_filenames[i],
            animal2_filename=animal2_filenames[i],
            recorded_animal=recorded_animals[i],
            trig_channelname=trig_channelnames[i],
        )


    # ---- Ditto ----
    neural_record_conditions = [
        '20260709_Ditto_MC_withCubone',
    ]
    dates_list = [
        '20260709_MC',
    ]
    videodates_list = [
        '20260709_MC',
    ]
    session_start_times = [
        10,
    ]
    trig_channelnames = [
        'Dev1/ai0',
                        ] # transcevier 1  is 'Dev1/ai0', and transceiver is 'Dev1/ai9'

    kilosortvers = [4] * len(dates_list)
    animal1_fixedorders = ['cubone'] * len(dates_list)
    animal2_fixedorders = ['ditto'] * len(dates_list)
    animal1_filenames = ["Cubone"] * len(dates_list)
    animal2_filenames = ["Ditto"] * len(dates_list)
    recorded_animals = ['ditto'] * len(dates_list) # make sure it's ditto


    for i, d in enumerate(neural_record_conditions):
        meta[d] = dict(
            neural_record_condition=neural_record_conditions[i],
            date_tgt=dates_list[i],
            videodate_tgt=videodates_list[i],
            kilosortver=kilosortvers[i],
            session_start_time=session_start_times[i],
            animal1_fixedorder=animal1_fixedorders[i],
            animal2_fixedorder=animal2_fixedorders[i],
            animal1_filename=animal1_filenames[i],
            animal2_filename=animal2_filenames[i],
            recorded_animal=recorded_animals[i],
            trig_channelname=trig_channelnames[i],
        )

    return meta

SESSION_METADATA = _build_session_metadata()



# ---------------------------------------------------------------------------
# core extraction
# ---------------------------------------------------------------------------

def _load_ni_data(date_tgt, animal1_filename, animal2_filename):
    """Locate and load the ni_data json used to align neural time to session
    start. Tries both animal-name orderings, matching the original notebook."""
    bhv_data_path = f"{bhv_data_root}{date_tgt}_{animal1_filename}_{animal2_filename}/"
    for a, b in [(animal2_filename, animal1_filename), (animal1_filename, animal2_filename)]:
        ni_data_json = glob.glob(bhv_data_path + f"{date_tgt}_{a}_{b}_ni_data_*.json")
        if ni_data_json:
            with open(ni_data_json[0]) as f:
                for line in f:
                    return json.loads(line)
    return None


def _get_trigger_elapsed_time(ni_data, trig_channelname):
    """Look up the elapsed_time for the given trigger channel name.

    Expected structure (kilosort4 sessions):
        ni_data['trigger_ts'] is a list of entries, each with a
        'channel_name' and a 'timepoints' list; we take the first
        timepoint's elapsed_time for the matching channel.
    """
    return next(
        entry['timepoints'][-1]['elapsed_time']
        for entry in ni_data['trigger_ts']
        if entry['channel_name'] == f"{trig_channelname}"
    )


def extract_spike_data(target_conditions=None, save=False):
    """
    Extract curated ('good'/'mua') spike times and cluster ids for each
    recording in `target_conditions`.

    Parameters
    ----------
    target_conditions : list of str, optional
        `neural_record_condition` values to process, matching the keys in
        SESSION_METADATA (e.g. "20260709_Cubone_MC_withDitto"). If None
        (default), processes every recording currently defined in
        SESSION_METADATA -- so you only need to edit the metadata dict
        above, not this call.
    save : bool
        If True, pickle the result dict to `data_saved_folder`.

    Returns
    -------
    dict : neural_record_condition -> spike data (see module docstring)
    """
    if target_conditions is None:
        target_conditions = list(SESSION_METADATA.keys())

    spike_data_all_conditions = {}

    for cond_tgt in target_conditions:
        if cond_tgt not in SESSION_METADATA:
            print(f"[skip] '{cond_tgt}' not found in SESSION_METADATA")
            continue

        info = SESSION_METADATA[cond_tgt]
        neural_record_condition = info['neural_record_condition']
        date_tgt = info['date_tgt']
        kilosortver = info['kilosortver']
        trig_channelname = info['trig_channelname']

        print(f"loading spike data for {date_tgt} ({neural_record_condition})")

        # ---- load spike sorting results ----
        ks_subdir = 'Kilosort' if kilosortver == 2 else 'kilosort4'
        base = f"{neural_data_folder}{neural_record_condition}/{ks_subdir}/"

        spike_time_data = np.load(base + 'spike_times.npy')
        spike_clusters_data = np.load(base + 'spike_clusters.npy')
        clusters_info_data = pd.read_csv(base + 'cluster_info.tsv', sep="\t")

        # ---- keep only manually curated units ----
        id_col = 'cluster_id' if 'cluster_id' in clusters_info_data.columns else 'id'
        good_clusters = clusters_info_data[
            (clusters_info_data.group == 'good') | (clusters_info_data.group == 'mua')
        ][id_col].values
        clusters_info_data = clusters_info_data[~pd.isnull(clusters_info_data.group)]

        keep = np.isin(spike_clusters_data, good_clusters)
        spike_time_data = spike_time_data[keep]
        spike_clusters_data = spike_clusters_data[keep]

        # ---- align to session start (best-effort) ----
        aligned = False
        try:
            ni_data = _load_ni_data(date_tgt, info['animal1_filename'], info['animal2_filename'])
            session_start_time_niboard_offset = ni_data['session_t0_offset']
            neural_start_time_niboard_offset = _get_trigger_elapsed_time(ni_data, trig_channelname)
            neural_start_time_session_start_offset = (
                neural_start_time_niboard_offset - session_start_time_niboard_offset
            )
            spike_time_sec = spike_time_data.ravel() / fs_spikes + neural_start_time_session_start_offset
            aligned = True
        except Exception as e:
            print(f"  [warn] could not align to session start ({e}); returning unaligned times")
            spike_time_sec = spike_time_data.ravel() / fs_spikes

        spike_data_all_conditions[cond_tgt] = dict(
            date_tgt=date_tgt,
            recorded_animal=info['recorded_animal'],
            spike_times_sec=spike_time_sec,
            spike_clusters=spike_clusters_data.ravel(),
            good_clusters=good_clusters,
            clusters_info=clusters_info_data,
            aligned=aligned,
        )

    if save:
        out_dir = os.path.join(data_saved_folder, 'spike_data_only')
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, 'spike_data_all_conditions.pkl')
        with open(out_path, 'wb') as f:
            pickle.dump(spike_data_all_conditions, f)
        print(f"saved to {out_path}")

    return spike_data_all_conditions


In [ ]:
if __name__ == "__main__":
    # no need to list anything here -- extract_spike_data() defaults to every
    # recording currently defined in SESSION_METADATA. Pass an explicit list
    # of neural_record_condition strings only if you want to run a subset.
    result = extract_spike_data(save=False)
    for cond, v in result.items():
        print(cond, "(", v['date_tgt'], v['recorded_animal'], ") -> n_spikes:", 
              len(v['spike_times_sec']), "n_units:", len(v['good_clusters']), "aligned:", v['aligned'])


## Load trial record data and get pull times (recorded animal / partner animal)

This follows the same convention as `BasicBhvAna_singlecam_wholebodylabels_allsessions_basicEvents.ipynb`:
- Load `TrialRecord`, `bhv_data`, and `session_info` json files for a session.
- Clean `trial_record` down to one row per real trial.
- Convert `bhv_data`'s trial-relative `time_points` to absolute session time using each trial's `trial_starttime`.
- Pull out `behavior_events==1` (lever1 pulls) and `behavior_events==2` (lever2 pulls) as pull times.

Because which animal is lever1 vs lever2 (and therefore "animal1"/"animal2") can flip from date to date, pull times are keyed off **which animal was actually recorded** for that session (`recorded_animal` in `SESSION_METADATA`) vs **its partner**, rather than a fixed animal1/animal2 order. `session_info`'s `lever1_animal`/`lever2_animal` fields (read fresh per date) are compared against `recorded_animal` to do this mapping.

In [ ]:
import warnings


def _load_trial_bhv_session(date_tgt, animal1_filename, animal2_filename):
    """Locate and load the TrialRecord / bhv_data / session_info json files
    for a session. Tries both animal-name orderings in the filename, same
    convention as `_load_ni_data` above and as the original bhv analysis
    notebook."""
    bhv_data_path = f"{bhv_data_root}{date_tgt}_{animal1_filename}_{animal2_filename}/"
    for a, b in [(animal2_filename, animal1_filename), (animal1_filename, animal2_filename)]:
        trial_record_json = glob.glob(bhv_data_path + f"{date_tgt}_{a}_{b}_TrialRecord_*.json")
        bhv_data_json = glob.glob(bhv_data_path + f"{date_tgt}_{a}_{b}_bhv_data_*.json")
        session_info_json = glob.glob(bhv_data_path + f"{date_tgt}_{a}_{b}_session_info_*.json")
        if trial_record_json and bhv_data_json and session_info_json:
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            return trial_record, bhv_data, session_info
    raise FileNotFoundError(
        f"could not find TrialRecord/bhv_data/session_info json for {date_tgt} in {bhv_data_path}"
    )


def _clean_trial_record(trial_record):
    """Keep only the first row per trial_number (same idea as
    `trial_record_clean` in the original notebook, but uses pd.concat
    instead of the removed DataFrame.append)."""
    warnings.filterwarnings('ignore')
    ntrials = int(np.max(trial_record['trial_number']))
    rows = [trial_record[trial_record['trial_number'] == itrial + 1].iloc[[0]] for itrial in range(ntrials)]
    trial_record_clean = pd.concat(rows, ignore_index=True)
    return trial_record_clean


def _add_absolute_time(bhv_data, trial_record_clean):
    """Convert bhv_data's trial-relative `time_points` to absolute session
    time, using each trial's `trial_starttime` from trial_record_clean."""
    time_points_new = np.zeros(bhv_data.shape[0])
    ntrials = int(np.max(trial_record_clean['trial_number']))
    for itrial in range(ntrials):
        ind = (bhv_data['trial_number'] == itrial + 1).values
        time_points_new[ind] = (
            bhv_data.loc[ind, 'time_points'] + trial_record_clean['trial_starttime'].iloc[itrial]
        )
    bhv_data = bhv_data.copy()
    bhv_data['time_points'] = time_points_new
    bhv_data = bhv_data[bhv_data['time_points'] != 0]
    return bhv_data


def extract_pull_times(target_conditions=None, save=False):
    """
    Load the trial record / bhv_data / session_info for each recording in
    `target_conditions` and extract pull times for the *recorded* animal and
    its *partner* (seconds, absolute session time -- i.e. trial_starttime +
    time_points).

    Note: which physical animal is "lever1" vs "lever2" (and therefore which
    one is animal1 vs animal2 in the fixed-order sense) can flip from date to
    date. So rather than keying off animal1_fixedorder/animal2_fixedorder,
    this reads `session_info`'s `lever1_animal`/`lever2_animal` for *that*
    session and compares each to `recorded_animal` (from SESSION_METADATA,
    i.e. which animal this neural recording actually belongs to) to figure
    out which lever/pull-stream is the recorded animal's and which is the
    partner's.

    Parameters
    ----------
    target_conditions : list of str, optional
        `neural_record_condition` values to process, matching the keys in
        SESSION_METADATA. If None (default), processes every recording
        currently defined in SESSION_METADATA.
    save : bool
        If True, pickle the result dict to `data_saved_folder`.

    Returns
    -------
    dict : neural_record_condition -> dict with keys
        'date_tgt', 'recorded_animal', 'partner_animal',
        'pulltime_recorded' (np.array, seconds),
        'pulltime_partner'  (np.array, seconds),
        'trial_record_clean', 'bhv_data', 'session_info'
    """
    if target_conditions is None:
        target_conditions = list(SESSION_METADATA.keys())

    pull_times_all_conditions = {}

    for cond_tgt in target_conditions:
        if cond_tgt not in SESSION_METADATA:
            print(f"[skip] '{cond_tgt}' not found in SESSION_METADATA")
            continue

        info = SESSION_METADATA[cond_tgt]
        date_tgt = info['date_tgt']
        animal1_filename = info['animal1_filename']
        animal2_filename = info['animal2_filename']
        recorded_animal = info['recorded_animal']

        print(f"loading trial record / bhv data for {date_tgt} ({cond_tgt})")

        try:
            trial_record, bhv_data, session_info = _load_trial_bhv_session(
                date_tgt, animal1_filename, animal2_filename
            )
        except Exception as e:
            print(f"  [warn] could not load trial record for '{cond_tgt}': {e}")
            continue

        # who is lever1 / lever2 in *this* session's task code -- this can
        # differ date to date, so always read it fresh from session_info
        animal_lever1 = session_info['lever1_animal'][0].lower()
        animal_lever2 = session_info['lever2_animal'][0].lower()

        # clean up trial_record (one row per real trial) and convert
        # bhv_data time_points to absolute session time
        trial_record_clean = _clean_trial_record(trial_record)
        bhv_data = _add_absolute_time(bhv_data, trial_record_clean)

        # behavior_events == 1 -> lever1 pull, == 2 -> lever2 pull
        pulltime_lever1 = np.array(bhv_data.loc[bhv_data['behavior_events'] == 1, 'time_points'])
        pulltime_lever2 = np.array(bhv_data.loc[bhv_data['behavior_events'] == 2, 'time_points'])

        # figure out which lever belongs to the recorded animal for *this*
        # session, and assign recorded/partner pull times accordingly
        if recorded_animal == animal_lever1:
            pulltime_recorded = pulltime_lever1
            pulltime_partner = pulltime_lever2
            partner_animal = animal_lever2
        elif recorded_animal == animal_lever2:
            pulltime_recorded = pulltime_lever2
            pulltime_partner = pulltime_lever1
            partner_animal = animal_lever1
        else:
            print(f"  [warn] recorded_animal '{recorded_animal}' matches neither "
                  f"lever1_animal ('{animal_lever1}') nor lever2_animal ('{animal_lever2}') "
                  f"for '{cond_tgt}'; skipping")
            continue

        pull_times_all_conditions[cond_tgt] = dict(
            date_tgt=date_tgt,
            recorded_animal=recorded_animal,
            partner_animal=partner_animal,
            pulltime_recorded=pulltime_recorded,
            pulltime_partner=pulltime_partner,
            trial_record_clean=trial_record_clean,
            bhv_data=bhv_data,
            session_info=session_info,
        )

    if save:
        out_dir = os.path.join(data_saved_folder, 'pull_time_only')
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, 'pull_times_all_conditions.pkl')
        with open(out_path, 'wb') as f:
            pickle.dump(pull_times_all_conditions, f)
        print(f"saved to {out_path}")

    return pull_times_all_conditions


In [ ]:
if __name__ == "__main__":
    # no need to list anything here -- extract_pull_times() defaults to every
    # recording currently defined in SESSION_METADATA. Pass an explicit list
    # of neural_record_condition strings only if you want to run a subset.
    pull_times_result = extract_pull_times(save=False)
    for cond, v in pull_times_result.items():
        print(cond, "(", v['date_tgt'], ") -> recorded:", v['recorded_animal'],
              "n_pulls:", len(v['pulltime_recorded']), " | partner:", v['partner_animal'],
              "n_pulls:", len(v['pulltime_partner']))


## PSTH of each good neuron around pulls (self and partner), one figure per session

For every session (`neural_record_condition`), and for every manually curated unit in `good_clusters`, this bins spikes in a `[-4, 4]` s window around each pull event and averages across pulls to get a mean firing-rate PSTH.

- Two separate figures per session: one aligned to the **recorded animal's own pulls**, one aligned to the **partner's pulls**.
- All good neurons for that session are overlaid as separate lines in the same figure.
- Bin size is a parameter (`binsize`), default **50 ms**.
- Figures are saved as PDFs into `figure_save_folder` (defaults to the folder this notebook is running from).

In [ ]:
import matplotlib.pyplot as plt

# folder where PSTH figures get saved as PDFs.
# os.getcwd() is the directory the Jupyter kernel was started in, which is
# normally the same folder as this notebook -- change this if that's not
# the case for you.
figure_save_folder = os.getcwd()


In [ ]:
from scipy.ndimage import gaussian_filter1d


def compute_psth_meanfr(spike_times, event_times, window=(-4, 4), binsize=0.05, smooth_sigma=0.1):
    """
    Trial-averaged PSTH (mean firing rate per bin, in Hz) for one neuron
    aligned to a set of event times (e.g. pulls).

    Smoothing (if smooth_sigma is not None) is applied to each trial's
    binned firing-rate trace *before* averaging across trials, so the
    returned SEM reflects the smoothed trial-to-trial variability and
    lines up correctly with the smoothed mean for error-bar shading.

    Parameters
    ----------
    spike_times : array-like
        Spike times (s) for a single neuron, same time base as event_times.
    event_times : array-like
        Event (e.g. pull) times (s) to align to.
    window : tuple
        (start, end) in seconds relative to the event, e.g. (-4, 4).
    binsize : float
        Bin width in seconds.
    smooth_sigma : float or None
        Gaussian smoothing sigma, in seconds. Set to None or 0 to disable
        smoothing entirely (raw binned PSTH).

    Returns
    -------
    bin_centers : np.array
    mean_fr : np.array   -- mean firing rate (Hz) per bin, averaged over events
    sem_fr : np.array    -- SEM across events, per bin
    """
    bins = np.arange(window[0], window[1] + binsize, binsize)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    spike_times = np.asarray(spike_times)
    event_times = np.asarray(event_times)

    if event_times.size == 0:
        nan_fill = np.full(len(bin_centers), np.nan)
        return bin_centers, nan_fill, nan_fill

    counts = np.zeros((len(event_times), len(bin_centers)))
    for i, t0 in enumerate(event_times):
        rel = spike_times[(spike_times >= t0 + window[0]) & (spike_times < t0 + window[1])] - t0
        counts[i], _ = np.histogram(rel, bins=bins)

    fr = counts / binsize  # Hz, per event, per bin

    # smooth each trial's trace first, then average -- keeps mean and SEM consistent
    if smooth_sigma:
        sigma_bins = smooth_sigma / binsize
        fr = gaussian_filter1d(fr, sigma=sigma_bins, axis=1, mode='nearest')

    mean_fr = np.nanmean(fr, axis=0)
    sem_fr = np.nanstd(fr, axis=0) / np.sqrt(fr.shape[0])
    return bin_centers, mean_fr, sem_fr


def plot_psth_all_neurons_per_session(spike_data_all_conditions, pull_times_all_conditions,
                                       window=(-4, 4), binsize=0.05, smooth_sigma=0.1,
                                       ncols=8, save_figures=True, figure_save_folder=figure_save_folder):
    """
    For every session present in both `spike_data_all_conditions` (from
    extract_spike_data()) and `pull_times_all_conditions` (from
    extract_pull_times()), plot the mean +/- SEM PSTH of every good/mua
    unit, one panel per unit, aligned separately to the recorded animal's
    own pulls and to the partner's pulls -- two figures per session.

    Parameters
    ----------
    window : tuple
        Time window around each pull, in seconds. Default (-4, 4).
    binsize : float
        PSTH bin width, in seconds. Default 0.05 (50 ms).
    smooth_sigma : float or None
        Gaussian smoothing sigma (s) applied to each trial's PSTH before
        averaging. Default 0.1 (100 ms). Set to None/0 for no smoothing.
    ncols : int
        Number of subplot columns (one panel per unit). Row count is
        derived from the number of units. Default 8.
    save_figures : bool
        If True, save each figure as a PDF into `figure_save_folder`.
    figure_save_folder : str
        Where to save PDFs. Defaults to the module-level `figure_save_folder`
        (the notebook's working directory).

    Returns
    -------
    dict : neural_record_condition -> {'recorded': (bin_centers, {cluster_id: (mean_fr, sem_fr)}),
                                        'partner':  (bin_centers, {cluster_id: (mean_fr, sem_fr)})}
    """
    common_conditions = [c for c in spike_data_all_conditions if c in pull_times_all_conditions]
    missing = [c for c in spike_data_all_conditions if c not in pull_times_all_conditions]
    if missing:
        print(f"[warn] no pull times found for: {missing} -- skipping those sessions")

    if save_figures and not os.path.exists(figure_save_folder):
        os.makedirs(figure_save_folder)

    psth_all_conditions = {}

    for cond in common_conditions:
        spikedata = spike_data_all_conditions[cond]
        pulldata = pull_times_all_conditions[cond]

        spike_times_sec = spikedata['spike_times_sec']
        spike_clusters = spikedata['spike_clusters']
        good_clusters = spikedata['good_clusters']
        nunits = len(good_clusters)

        align_options = [
            ('recorded', pulldata['pulltime_recorded'], pulldata['recorded_animal']),
            ('partner', pulldata['pulltime_partner'], pulldata['partner_animal']),
        ]

        psth_all_conditions[cond] = {}

        for align_label, event_times, animal_name in align_options:
            nrows = int(np.ceil(nunits / ncols)) if nunits > 0 else 1
            fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 1.8),
                                      sharex=True, sharey=False, squeeze=False)
            axes_flat = axes.flatten()

            per_cluster_psth = {}
            bin_centers = None

            for ci, clu in enumerate(good_clusters):
                ax = axes_flat[ci]
                clu_spike_times = spike_times_sec[spike_clusters == clu]
                bin_centers, mean_fr, sem_fr = compute_psth_meanfr(
                    clu_spike_times, event_times, window=window, binsize=binsize, smooth_sigma=smooth_sigma
                )
                per_cluster_psth[clu] = (mean_fr, sem_fr)

                ax.plot(bin_centers, mean_fr, color='steelblue', linewidth=1)
                ax.fill_between(bin_centers, mean_fr - sem_fr, mean_fr + sem_fr,
                                 color='steelblue', alpha=0.3, linewidth=0)
                ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
                ax.set_title(f"unit {clu}", fontsize=8)
                ax.tick_params(axis='both', labelsize=6)

            # turn off any unused panels
            for j in range(nunits, len(axes_flat)):
                axes_flat[j].axis('off')

            fig.suptitle(f"{cond} -- aligned to {animal_name}'s pulls ({align_label}, "
                         f"n={len(event_times)} pulls, n={nunits} units)", fontsize=11)
            fig.text(0.5, 0.0, "Time from pull (s)", ha='center', fontsize=10)
            fig.text(0.0, 0.5, "Firing rate (Hz)", va='center', rotation='vertical', fontsize=10)
            plt.tight_layout(rect=[0.02, 0.02, 1, 0.96])

            psth_all_conditions[cond][align_label] = (bin_centers, per_cluster_psth)

            if save_figures:
                fname = f"{cond}_psth_{align_label}Pulls_bin{int(binsize*1000)}ms.pdf"
                fig.savefig(os.path.join(figure_save_folder, fname), format='pdf')
                print(f"saved {os.path.join(figure_save_folder, fname)}")

            plt.show()

    return psth_all_conditions

In [ ]:
if __name__ == "__main__":
    spike_data_all_conditions = extract_spike_data(save=False)
    pull_times_all_conditions = extract_pull_times(save=False)

    # binsize is optional -- defaults to 0.05 (50 ms); window defaults to (-4, 4) s
    psth_all_conditions = plot_psth_all_neurons_per_session(
        spike_data_all_conditions, pull_times_all_conditions,
        window=(-4, 4), binsize=0.05, save_figures=True,
    )
